# Python für Verfahrenstechniker — Teil 2
## Kontrollstrukturen, Funktionen und Fehlersuche

In Teil 1 haben Sie Python als besseren Taschenrechner benutzt: Jede Zeile wurde genau einmal
von oben nach unten ausgeführt. In diesem Teil lernen Sie, wie ein Programm **entscheidet**
(`if`), **wiederholt** (`for`, `while`) und wie Sie eigene **Funktionen** schreiben.

---

### Lernziele

Nach diesem Notebook können Sie

- Fallunterscheidungen mit `if` / `elif` / `else` programmieren,
- Schleifen mit `for` und `while` schreiben und den Unterschied begründen,
- eigene Funktionen mit Parametern, Standardwerten und Rückgabewerten definieren,
- eine Fehlermeldung (Traceback) lesen und die häufigsten Fehlertypen einordnen.

**Voraussetzung:** Teil 1. **Zeitaufwand:** ca. 90 Minuten.

---
## 1. Fallunterscheidung: if / elif / else

Ein Strömungszustand hängt von der Reynolds-Zahl ab. Diese Fallunterscheidung schreibt man so:

In [1]:
Re = 4200

if Re < 2300:
    print("laminar")
elif Re < 10000:
    print("Übergangsbereich")
else:
    print("turbulent")

Übergangsbereich


### Die Einrückung ist Teil der Sprache

Andere Sprachen benutzen geschweifte Klammern, um zusammengehörige Anweisungen zu markieren.
Python benutzt dafür die **Einrückung**. Alles, was gleich weit eingerückt ist, gehört zum selben
Block. Der Standard sind **vier Leerzeichen** pro Ebene.

```python
if Bedingung:          # Doppelpunkt am Zeilenende nicht vergessen!
    anweisung_1        # gehört zum if
    anweisung_2        # gehört zum if
anweisung_3            # wird immer ausgeführt
```

Vergessen Sie den Doppelpunkt oder die Einrückung, bekommen Sie einen `SyntaxError` bzw.
einen `IndentationError`. Diese beiden Fehler werden Sie in der ersten Woche oft sehen — das ist normal.

### Mehrere Bedingungen verknüpfen

Mit `and`, `or` und `not` kombinieren Sie Wahrheitswerte.

In [2]:
T = 385.0     # K
p = 1.2e5     # Pa

if T > 373.15 and p < 2e5:
    print("Wasser läge unter diesen Bedingungen dampfförmig vor.")

if not (2300 < Re < 10000):
    print("Re liegt nicht im Übergangsbereich.")

Wasser läge unter diesen Bedingungen dampfförmig vor.


Beachten Sie die Schreibweise `2300 < Re < 10000` — solche verketteten Vergleiche sind in Python
erlaubt und lesen sich wie in der Mathematik.

---
## 2. Die for-Schleife

Eine `for`-Schleife arbeitet **jedes Element einer Liste** nacheinander ab. Das ist der Normalfall
in der Praxis: dieselbe Rechnung für viele Messstellen, Stoffe oder Zeitpunkte.

In [3]:
temperaturen = [78.4, 82.1, 91.5, 97.3, 99.8]   # °C

for T in temperaturen:
    T_kelvin = T + 273.15
    print(f"{T:6.1f} °C  =  {T_kelvin:7.2f} K")

  78.4 °C  =   351.55 K
  82.1 °C  =   355.25 K
  91.5 °C  =   364.65 K
  97.3 °C  =   370.45 K
  99.8 °C  =   372.95 K


Die Laufvariable — hier `T` — nimmt in jedem Durchlauf den Wert des nächsten Listenelements an.
Der eingerückte Block wird einmal pro Element ausgeführt.

### Mit Zählindex: enumerate

Brauchen Sie zusätzlich die Position, verwenden Sie `enumerate`:

In [4]:
for i, T in enumerate(temperaturen):
    print(f"Messstelle {i + 1}: {T:.1f} °C")

Messstelle 1: 78.4 °C
Messstelle 2: 82.1 °C
Messstelle 3: 91.5 °C
Messstelle 4: 97.3 °C
Messstelle 5: 99.8 °C


### Über einen Zahlenbereich: range

`range(n)` erzeugt die Zahlen `0, 1, ..., n-1`. `range(start, stop, schritt)` erlaubt mehr Kontrolle.

In [5]:
for stufe in range(1, 6):        # 1 bis 5
    print(f"Trennstufe {stufe}")

Trennstufe 1
Trennstufe 2
Trennstufe 3
Trennstufe 4
Trennstufe 5


### Das Akkumulator-Muster

Ein sehr häufiges Muster: Vor der Schleife eine Variable auf null setzen, in der Schleife etwas
dazuaddieren. So bilanzieren Sie zum Beispiel Massenströme.

In [6]:
massenstroeme = [12.4, 3.8, 0.9, 7.2]   # kg/h, vier Zuläufe

summe = 0.0                              # Startwert vor der Schleife
for m_punkt in massenstroeme:
    summe = summe + m_punkt              # kurz: summe += m_punkt

print(f"Gesamter Zulauf: {summe:.1f} kg/h")

Gesamter Zulauf: 24.3 kg/h


### Über ein Dictionary laufen

In [7]:
molare_massen = {"Wasser": 18.015, "Ethanol": 46.069, "Benzol": 78.114}

for stoff, M in molare_massen.items():
    n = 1000.0 / M      # Stoffmenge in 1 kg
    print(f"{stoff:10s}  M = {M:7.3f} g/mol  ->  n(1 kg) = {n:6.2f} mol")

Wasser      M =  18.015 g/mol  ->  n(1 kg) =  55.51 mol
Ethanol     M =  46.069 g/mol  ->  n(1 kg) =  21.71 mol
Benzol      M =  78.114 g/mol  ->  n(1 kg) =  12.80 mol


---
## 3. Die while-Schleife

Die `for`-Schleife brauchen Sie, wenn die Anzahl der Durchläufe vorher feststeht. Die
`while`-Schleife läuft, **solange eine Bedingung erfüllt ist** — typisch für iterative Verfahren,
bei denen Sie vorher nicht wissen, wie viele Schritte bis zur Konvergenz nötig sind.

Beispiel: Ein Behälter wird mit konstantem Volumenstrom befüllt. Wie viele Minuten dauert es,
bis 80 % Füllstand erreicht sind?

In [8]:
V_behaelter = 2.0        # m^3
V_punkt     = 0.05       # m^3/min
V           = 0.0        # aktueller Inhalt in m^3
t           = 0          # min

while V < 0.8 * V_behaelter:
    V = V + V_punkt
    t = t + 1

print(f"Nach {t} min sind {V:.2f} m^3 im Behälter ({V / V_behaelter * 100:.0f} % Füllstand).")

Nach 32 min sind 1.60 m^3 im Behälter (80 % Füllstand).


> **Gefahr Endlosschleife:** Wenn die Bedingung nie falsch wird, läuft die Schleife ewig. Achten
> Sie darauf, dass in jedem Durchlauf tatsächlich etwas verändert wird, das die Bedingung
> beeinflusst. Falls es doch passiert: das schwarze Quadrat (Stopp) in der Werkzeugleiste
> unterbricht die Ausführung.

Bei iterativen Rechnungen baut man zusätzlich immer einen **Notausgang** über eine maximale
Schrittzahl ein:

In [9]:
# Iterative Lösung von x = cos(x) (Fixpunktiteration)
import math

x = 1.0
schritt = 0
max_schritte = 100

while abs(math.cos(x) - x) > 1e-8 and schritt < max_schritte:
    x = math.cos(x)
    schritt += 1

print(f"Lösung x = {x:.8f} nach {schritt} Iterationen")

Lösung x = 0.73908513 nach 45 Iterationen


Solche Iterationsverfahren systematisch aufzustellen — Konvergenzordnung, Abbruchkriterien,
Fehlerabschätzung — ist Thema der Numerik-Veranstaltung. Hier reicht das Prinzip.

---
## 4. Funktionen

Sobald Sie eine Rechnung zum zweiten Mal brauchen, sollten Sie sie in eine **Funktion** packen.
Copy-Paste ist der Anfang jedes unwartbaren Programms: Wenn Sie später einen Fehler finden,
müssen Sie ihn an fünf Stellen korrigieren — und vergessen die sechste.

Aufbau einer Funktion:

```python
def name(parameter1, parameter2):
    """Kurzbeschreibung, was die Funktion tut."""
    ...Rechnung...
    return ergebnis
```

In [10]:
def reynolds(rho, w, d, eta):
    """Berechnet die Reynolds-Zahl einer Rohrströmung.

    Parameter
    ---------
    rho : Dichte in kg/m^3
    w   : mittlere Strömungsgeschwindigkeit in m/s
    d   : Innendurchmesser in m
    eta : dynamische Viskosität in Pa s

    Rückgabe
    --------
    Reynolds-Zahl (dimensionslos)
    """
    return rho * w * d / eta

Die Zelle oben erzeugt nur die Funktion, sie rechnet noch nichts. Erst der **Aufruf** führt sie aus:

In [11]:
Re_1 = reynolds(998.0, 1.2, 0.025, 1.002e-3)
Re_2 = reynolds(998.0, 0.3, 0.010, 1.002e-3)

print(f"Rohr 1: Re = {Re_1:.0f}")
print(f"Rohr 2: Re = {Re_2:.0f}")

Rohr 1: Re = 29880
Rohr 2: Re = 2988


Der Text in dreifachen Anführungszeichen ist der **Docstring**. Er dokumentiert die Funktion —
insbesondere die erwarteten **Einheiten**, was in der Verfahrenstechnik der wichtigste Teil ist.
Python zeigt ihn auf Anfrage an:

In [12]:
help(reynolds)

Help on function reynolds in module __main__:

reynolds(rho, w, d, eta)
    Berechnet die Reynolds-Zahl einer Rohrströmung.

    Parameter
    ---------
    rho : Dichte in kg/m^3
    w   : mittlere Strömungsgeschwindigkeit in m/s
    d   : Innendurchmesser in m
    eta : dynamische Viskosität in Pa s

    Rückgabe
    --------
    Reynolds-Zahl (dimensionslos)



### Standardwerte

Parameter können Standardwerte bekommen. Sie müssen dann beim Aufruf nicht angegeben werden.

In [13]:
def dichte_ideales_gas(p, T, M, R=8.314):
    """Dichte eines idealen Gases in kg/m^3.

    p in Pa, T in K, M in kg/mol, R in J/(mol K).
    """
    return p * M / (R * T)


print(f"{dichte_ideales_gas(1e5, 293.15, 28.96e-3):.4f} kg/m^3   (Luft, Normbedingungen)")
print(f"{dichte_ideales_gas(5e5, 350.00, 28.96e-3):.4f} kg/m^3   (Luft, 5 bar, 350 K)")

1.1882 kg/m^3   (Luft, Normbedingungen)
4.9761 kg/m^3   (Luft, 5 bar, 350 K)


Beim Aufruf können Sie die Parameter auch mit Namen angeben. Das ist bei vielen Parametern
deutlich lesbarer und schützt vor Verwechslungen:

In [14]:
rho = dichte_ideales_gas(p=1e5, T=293.15, M=28.96e-3)
print(f"{rho:.4f} kg/m^3")

1.1882 kg/m^3


### Beispiel: Antoine-Gleichung

Der Dampfdruck eines Reinstoffs wird häufig über die Antoine-Gleichung beschrieben:

$$\log_{10}\left(\frac{p^s}{\mathrm{mmHg}}\right) = A - \frac{B}{C + \vartheta/^\circ\mathrm{C}}$$

Die Konstanten A, B, C sind stoffspezifisch und gelten nur in einem begrenzten Temperaturbereich.

In [15]:
def dampfdruck_antoine(theta, A, B, C):
    """Dampfdruck nach Antoine.

    theta : Temperatur in °C
    A, B, C : Antoine-Konstanten für p in mmHg und theta in °C

    Rückgabe: Dampfdruck in Pa
    """
    p_mmHg = 10 ** (A - B / (C + theta))
    p_Pa = p_mmHg * 133.322            # Umrechnung mmHg -> Pa
    return p_Pa


# Antoine-Konstanten für Wasser (gültig ca. 1-100 °C)
A_w, B_w, C_w = 8.07131, 1730.63, 233.426

for theta in [20, 40, 60, 80, 100]:
    p_s = dampfdruck_antoine(theta, A_w, B_w, C_w)
    print(f"theta = {theta:3d} °C   ->   p_s = {p_s/1000:8.2f} kPa")

theta =  20 °C   ->   p_s =     2.33 kPa
theta =  40 °C   ->   p_s =     7.36 kPa
theta =  60 °C   ->   p_s =    19.87 kPa
theta =  80 °C   ->   p_s =    47.27 kPa
theta = 100 °C   ->   p_s =   101.34 kPa


Bei 100 °C kommt ungefähr der Normdruck von 101,3 kPa heraus — ein guter Plausibilitätstest.
**Prüfen Sie jede neue Funktion an einem Wert, den Sie kennen.** Das ist die wirksamste einzelne
Maßnahme gegen Rechenfehler.

### Mehrere Rückgabewerte

Eine Funktion darf mehrere Werte zurückgeben, getrennt durch Kommata:

In [16]:
def stroemungszustand(rho, w, d, eta):
    """Gibt Reynolds-Zahl und Bezeichnung des Strömungszustands zurück."""
    Re = rho * w * d / eta

    if Re < 2300:
        zustand = "laminar"
    elif Re < 10000:
        zustand = "Übergangsbereich"
    else:
        zustand = "turbulent"

    return Re, zustand


Re, zustand = stroemungszustand(998.0, 1.2, 0.025, 1.002e-3)
print(f"Re = {Re:.0f}  ->  {zustand}")

Re = 29880  ->  turbulent


### Variablen innerhalb einer Funktion

Variablen, die in einer Funktion entstehen, existieren nur dort. `p_mmHg` aus der Antoine-Funktion
ist außerhalb nicht verfügbar:

In [17]:
# Diese Zelle erzeugt absichtlich einen Fehler - führen Sie sie aus und lesen Sie die Meldung.
print(p_mmHg)

NameError: name 'p_mmHg' is not defined

Das ist kein Mangel, sondern der Sinn der Sache: Eine Funktion ist eine abgeschlossene Einheit.
Alles, was sie braucht, kommt über die Parameter hinein; alles, was sie liefert, kommt über
`return` heraus. Damit können Sie sie überall wiederverwenden, ohne Nebenwirkungen zu befürchten.

---
## 5. Fehlermeldungen lesen

Fehlermeldungen sind keine Strafe, sondern die beste Diagnose, die Sie bekommen können. Ein
**Traceback** liest man **von unten nach oben**:

- die **letzte Zeile** nennt den Fehlertyp und die Ursache,
- darüber steht, in welcher Zeile der Fehler auftrat,
- bei verschachtelten Funktionsaufrufen darüber der Weg dorthin.

Die häufigsten Fehlertypen im Überblick:

| Fehlertyp | Ursache | Typischer Auslöser |
|-----------|---------|--------------------|
| `SyntaxError` | Python versteht die Zeile nicht | fehlender Doppelpunkt, unbalancierte Klammer |
| `IndentationError` | Einrückung passt nicht | Tab und Leerzeichen gemischt |
| `NameError` | Name unbekannt | Tippfehler, Zelle nicht ausgeführt |
| `TypeError` | Operation für diesen Typ nicht definiert | Text plus Zahl addiert |
| `ValueError` | Typ passt, Wert nicht | `float("1,5")` mit Komma |
| `IndexError` | Index außerhalb der Liste | ab 1 statt ab 0 gezählt |
| `KeyError` | Schlüssel nicht im Dictionary | Tippfehler im Stoffnamen |
| `ZeroDivisionError` | Division durch null | Nenner wird im Grenzfall null |

Führen Sie die folgenden Zellen aus und ordnen Sie die Meldungen zu.

In [18]:
werte = [1.0, 2.0, 3.0]
print(werte[3])          # -> IndexError

IndexError: list index out of range

In [19]:
M = {"Wasser": 18.015}
print(M["wasser"])       # -> KeyError (Groß-/Kleinschreibung!)

KeyError: 'wasser'

In [20]:
print(float("1,5"))      # -> ValueError (Komma statt Punkt)

ValueError: could not convert string to float: '1,5'

In [21]:
print("Temperatur: " + 350)     # -> TypeError

TypeError: can only concatenate str (not "int") to str

Die letzte Zeile korrigiert man entweder mit `str(350)` oder — besser — mit einem f-String:
`print(f"Temperatur: {350}")`.

> **Vorgehen bei einem Fehler:**
> 1. letzte Zeile der Meldung lesen — welcher Fehlertyp?
> 2. Zeilennummer suchen — wo genau?
> 3. beteiligte Variablen mit `print()` oder `type()` prüfen — was steht wirklich drin?
>
> Erst danach lohnt es sich zu suchen. Raten kostet immer mehr Zeit als Nachschauen.

---
## 6. Übungen

### Übung 2.1 — Ventilstellung

Schreiben Sie eine Fallunterscheidung, die für einen gemessenen Druck `p` (in bar) ausgibt:

- unter 1,0 bar: `"Druck zu niedrig - Ventil schließen"`
- zwischen 1,0 und 6,0 bar: `"Normalbetrieb"`
- über 6,0 bar: `"Druck zu hoch - Sicherheitsventil öffnen"`

Testen Sie Ihre Lösung mit `p = 0.8`, `p = 3.5` und `p = 7.2`.

In [22]:
# Ihre Lösung:
p = 3.5   # bar

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
p = 3.5   # bar

if p < 1.0:
    print("Druck zu niedrig - Ventil schließen")
elif p <= 6.0:
    print("Normalbetrieb")
else:
    print("Druck zu hoch - Sicherheitsventil öffnen")
```

</details>

### Übung 2.2 — Tabelle der Umrechnung

Geben Sie mit einer `for`-Schleife eine Tabelle aus, die für die Temperaturen 0, 25, 50, 75 und
100 °C jeweils den Wert in Kelvin und in Grad Fahrenheit enthält
($\vartheta_F = \vartheta_C \cdot 1{,}8 + 32$). Nutzen Sie `range` und formatieren Sie die Spalten
mit fester Breite.

In [23]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
print(f"{'°C':>6s} {'K':>10s} {'°F':>8s}")
print("-" * 26)

for theta in range(0, 101, 25):
    T_K = theta + 273.15
    theta_F = theta * 1.8 + 32
    print(f"{theta:6d} {T_K:10.2f} {theta_F:8.1f}")
```

</details>

### Übung 2.3 — Funktion für den Druckverlust

Schreiben Sie eine Funktion `druckverlust(rho, w, d, L, lam)`, die den Druckverlust in einem
geraden Rohr nach Darcy-Weisbach berechnet:

$$\Delta p = \lambda \, \frac{L}{d} \, \frac{\rho \, w^2}{2}$$

Der Rohrreibungsbeiwert `lam` soll standardmäßig 0,03 betragen. Schreiben Sie einen Docstring mit
allen Einheiten. Testen Sie mit ρ = 998 kg/m³, w = 1,2 m/s, d = 0,025 m, L = 15 m.

In [24]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
def druckverlust(rho, w, d, L, lam=0.03):
    """Druckverlust einer Rohrströmung nach Darcy-Weisbach.

    rho : Dichte in kg/m^3
    w   : mittlere Geschwindigkeit in m/s
    d   : Innendurchmesser in m
    L   : Rohrlänge in m
    lam : Rohrreibungsbeiwert (dimensionslos), Standardwert 0.03

    Rückgabe: Druckverlust in Pa
    """
    return lam * L / d * rho * w ** 2 / 2


dp = druckverlust(998.0, 1.2, 0.025, 15.0)
print(f"Druckverlust: {dp:.0f} Pa = {dp/1e5:.4f} bar")
```

</details>

### Übung 2.4 — Schleife und Funktion kombinieren

Nutzen Sie die Funktion `dampfdruck_antoine` von oben und die Antoine-Konstanten für **Ethanol**
(A = 8,20417, B = 1642,89, C = 230,300, gültig für p in mmHg und ϑ in °C).

Geben Sie mit einer Schleife über 20, 30, ..., 80 °C den Dampfdruck von Wasser und Ethanol
nebeneinander in einer Tabelle aus. Bei welcher Temperatur ist der Dampfdruck von Ethanol etwa
doppelt so hoch wie der von Wasser?

In [25]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
A_w, B_w, C_w = 8.07131, 1730.63, 233.426     # Wasser
A_e, B_e, C_e = 8.20417, 1642.89, 230.300     # Ethanol

print(f"{'°C':>5s} {'p_Wasser/kPa':>14s} {'p_Ethanol/kPa':>15s} {'Verhältnis':>12s}")
print("-" * 49)

for theta in range(20, 81, 10):
    p_w = dampfdruck_antoine(theta, A_w, B_w, C_w)
    p_e = dampfdruck_antoine(theta, A_e, B_e, C_e)
    print(f"{theta:5d} {p_w/1000:14.2f} {p_e/1000:15.2f} {p_e/p_w:12.2f}")

# Das Verhältnis liegt im gesamten Bereich bei etwa 2,2-2,4 und nimmt
# mit steigender Temperatur leicht ab.
```

</details>

### Übung 2.5 — Iteration

Ein Rührkessel mit 500 L Inhalt wird bei 90 °C betrieben und kühlt pro Minute um 3 % der
Temperaturdifferenz zur Umgebung (20 °C) ab:

$$\vartheta_{neu} = \vartheta - 0{,}03 \, (\vartheta - \vartheta_U)$$

Schreiben Sie eine `while`-Schleife, die zählt, wie viele Minuten es dauert, bis die Temperatur
unter 40 °C fällt. Bauen Sie einen Notausgang bei 1000 Schritten ein.

In [26]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
theta   = 90.0     # °C
theta_U = 20.0     # °C
t = 0              # min

while theta > 40.0 and t < 1000:
    theta = theta - 0.03 * (theta - theta_U)
    t += 1

print(f"Nach {t} min ist die Temperatur auf {theta:.2f} °C gefallen.")
```

</details>

---
## Zusammenfassung

| Konstrukt | Kurzform |
|-----------|----------|
| Fallunterscheidung | `if a < b:` ... `elif ...:` ... `else:` |
| Schleife über Liste | `for x in liste:` |
| Schleife mit Index | `for i, x in enumerate(liste):` |
| Zahlenbereich | `for i in range(1, 6):` |
| bedingte Schleife | `while bedingung:` |
| Funktion | `def f(a, b=1.0):` ... `return ergebnis` |
| Dokumentation | Docstring in `""" ... """`, abrufbar mit `help(f)` |

**Merksätze:**

- Doppelpunkt und Einrückung sind Pflicht, nicht Geschmackssache.
- Jede `while`-Schleife braucht ein Abbruchkriterium und einen Notausgang.
- Jede neue Funktion an einem bekannten Wert testen.
- Fehlermeldungen von unten nach oben lesen.

---

**Weiter mit Teil 3:** NumPy und grafische Darstellung.